# Does a product carry some seconds twice?

**Where a flight's raw files overlap, a stitched per-second product can carry the overlapping seconds twice, with two different values for one second and a row count that still looks right.** In the synthetic flight below, ten seconds are written twice, all ten pairs of copies differ, and keeping both rows adds 781 counts to the flight sum. Three checks on the time column alone (duplicate timestamps, backward steps, gaps) find the overlap, and a comparison of the copies names the rule a downstream product applied.

**What you get**

- the duplicated seconds, the backward step and any gap beside them;
- what keep-first, keep-later and the mean of the copies each do to a flight sum;
- a test of which copy a downstream product kept, and a cell for your own per-second files.

> **Every number in this notebook is synthetic**, drawn from a fixed-seed random-number generator in section 2; no data file is read.

**Before this:** [05_triage_and_gallery](../1_getting_started/05_triage_and_gallery.ipynb) (Looking at particles: a contact sheet, a gallery, and the triage page). **Related:** [03_clock_offsets](03_clock_offsets.ipynb). **Needs:** core package only (`pip install "arrayprobe[zarr]"` adds an optional section).

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/5_data_quality/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## 1. Why a duplicated second is easy to miss

A per-second series with N rows looks complete when N matches the flight's duration. An overlap of k
seconds adds k rows, so a file with k extra rows and k missing seconds elsewhere has exactly the right
length. Counting rows is not a check; counting **distinct** seconds, and the steps between them, is.

## 2. Two synthetic raw files that overlap

A 3600 s flight. File A covers seconds 0-1809; file B starts at second 1800, so seconds 1800-1809 appear
in both. The concentration is a smooth cloud signal times Poisson noise; the overlapping seconds are
drawn independently in each file, as two writers decoding different buffers would. A five-second gap
(2500-2504) is cut from file B so the gap check has something to find.

In [2]:
import numpy as np

rng = np.random.default_rng(27)
T = 3600
t_all = np.arange(T)
signal = 80.0 + 60.0 * np.sin(2 * np.pi * t_all / 1200.0) ** 2   # count/L, cloudy pattern

def draw(seconds):
    return rng.poisson(signal[seconds]).astype(float)

t_a = np.arange(0, 1810)                       # file A: 0..1809
t_b = np.arange(1800, T)                       # file B: 1800..3599 (overlap 1800..1809)
t_b = t_b[(t_b < 2500) | (t_b > 2504)]         # a five-second gap in file B
v_a = draw(t_a)
v_b = draw(t_b)

# The stitched product: file A then file B, as a writer that concatenates would emit it.
time = np.concatenate([t_a, t_b])
value = np.concatenate([v_a, v_b])
print("rows", time.size, " distinct seconds", np.unique(time).size, " flight length", T)

rows 3605  distinct seconds 3595  flight length 3600


## 3. Detection

Three checks on the time column alone. Each returns the seconds it flags, so a long flight prints a
short list.

In [3]:
def check_duplicates(time):
    """Seconds that appear more than once, with their count."""
    u, n = np.unique(time, return_counts=True)
    return [(float(s), int(c)) for s, c in zip(u[n > 1], n[n > 1])]

def check_backsteps(time):
    """Row indices where time does not increase, with the size of the step."""
    d = np.diff(time)
    idx = np.flatnonzero(d <= 0)
    return [(int(i + 1), float(d[i])) for i in idx]

def check_gaps(time):
    """Runs of missing seconds between the first and last second, as (first, last missing)."""
    u = np.unique(time)
    d = np.diff(u)
    idx = np.flatnonzero(d > 1)
    return [(float(u[i] + 1), float(u[i + 1] - 1)) for i in idx]

dups = check_duplicates(time)
back = check_backsteps(time)
gaps = check_gaps(time)
print("duplicated seconds:", len(dups), "from", dups[0][0], "to", dups[-1][0])
print("backward steps (row, step):", back)
print("gaps (first, last missing second):", gaps)
assert len(dups) == 10 and back == [(1810, -9.0)] and gaps == [(2500.0, 2504.0)]

duplicated seconds: 10 from 1800.0 to 1809.0
backward steps (row, step): [(1810, -9.0)]
gaps (first, last missing second): [(2500.0, 2504.0)]


The single backward step of -9 is the signature: the series reaches second 1809, steps back to 1800,
and re-advances through ten seconds it has already written.

**Do the two copies agree?** If they are identical the choice of copy cannot matter. Compare them.

In [4]:
first = {}
last = {}
for s, v in zip(time, value):
    first.setdefault(float(s), v)
    last[float(s)] = v
dup_secs = [s for s, _ in dups]
print("second  first copy  later copy")
for s in dup_secs:
    print("%6d  %10.0f  %10.0f" % (s, first[s], last[s]))
n_differ = sum(first[s] != last[s] for s in dup_secs)
print("copies that differ:", n_differ, "of", len(dup_secs))

second  first copy  later copy
  1800          91          68
  1801          91          80
  1802          68          52
  1803          98          85
  1804          80          84
  1805          77          70
  1806          75          80
  1807          65          88
  1808          59          75
  1809          95          99
copies that differ: 10 of 10


## 4. What the choice of copy does

Keep-first, keep-last and the mean of the two each give one value per second. The flight sum (or any
average over a leg that contains the overlap) moves by the difference on those seconds only. Keeping
both rows double-counts them, which is the one choice that is always wrong for a per-second product.

In [5]:
secs = np.array(sorted(first))
keep_first = np.array([first[s] for s in secs])
keep_last = np.array([last[s] for s in secs])
mean_both = 0.5 * (keep_first + keep_last)

print("%-22s %12s" % ("rule", "flight sum"))
for name, x in [("keep both rows", value), ("keep first", keep_first),
                ("keep later", keep_last), ("mean of the two", mean_both)]:
    print("%-22s %12.0f" % (name, x.sum()))
on_dup = np.isin(secs, dup_secs)
print("keep-later minus keep-first, on the duplicated seconds only: %.0f"
      % (keep_last[on_dup].sum() - keep_first[on_dup].sum()))
assert value.sum() > keep_last.sum()

rule                     flight sum
keep both rows               396137
keep first                   395356
keep later                   395338
mean of the two              395347
keep-later minus keep-first, on the duplicated seconds only: -18


## 5. Which rule did a downstream product apply?

When a later product holds one row per second, compare it with each candidate on the duplicated seconds
only: the rule it applied matches on every one of them, and the other rule matches only where the two
copies happen to agree. Here a stand-in "downstream product" is built with keep-later, and the test
recovers that.

In [6]:
downstream = dict(zip(secs.tolist(), keep_last))   # stand-in for a product you were handed

def which_copy(downstream, first, last, dup_secs):
    m_first = sum(downstream.get(s) == first[s] for s in dup_secs)
    m_last = sum(downstream.get(s) == last[s] for s in dup_secs)
    return m_first, m_last

m_first, m_last = which_copy(downstream, first, last, dup_secs)
print("matches keep-first: %d of %d;  matches keep-later: %d of %d"
      % (m_first, len(dup_secs), m_last, len(dup_secs)))
assert m_last == len(dup_secs)

matches keep-first: 0 of 10;  matches keep-later: 10 of 10


**Where to look first.** The overlap usually starts one second after a raw file's start time (the
second encoded in its name or header): the writer emitted the tail of the previous file and the head of
the next for the same seconds. List your raw files' start seconds next to the duplicated block.

## 6. Check your own data

Set `SERIES` to a per-second file and its time variable (seconds of day or since start), and
optionally `VALUE_VAR` to a variable to compare the copies on. With `SERIES = None` the cell runs on the
synthetic flight above. Midnight crossings show as a large backward step; unwrap them before reading the
table.

In [7]:
SERIES = None       # e.g. ("my_probe_1Hz.nc", "time")
VALUE_VAR = None    # e.g. "total_concentration"

def load(path, var):
    import xarray as xr
    with xr.open_dataset(path, decode_times=False) as ds:
        return np.asarray(ds[var].values, float)

if SERIES:
    t = load(*SERIES)
    v = load(SERIES[0], VALUE_VAR) if VALUE_VAR else None
else:
    t, v = time.astype(float), value

d, b, g = check_duplicates(t), check_backsteps(t), check_gaps(t)
print("rows %d, distinct seconds %d" % (t.size, np.unique(t).size))
print("duplicated seconds: %d  backward steps: %d  gaps: %d" % (len(d), len(b), len(g)))
for row, step in b:
    print("  step back of %g s at row %d (time %g)" % (step, row, t[row]))
if v is not None and d:
    f, l = {}, {}
    for s, x in zip(t.tolist(), v):
        f.setdefault(s, x)
        l[s] = x
    differ = sum(f[s] != l[s] for s, _ in d)
    print("copies that differ: %d of %d" % (differ, len(d)))
assert SERIES or (len(d), len(b), len(g)) == (10, 1, 1)

rows 3605, distinct seconds 3595
duplicated seconds: 10  backward steps: 1  gaps: 1
  step back of -9 s at row 1810 (time 1800)
copies that differ: 10 of 10
